<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/05-agents/04-mcp-and-the-tool-ecosystem.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCP 与工具生态（概念）

**目标：** 理解 Model Context Protocol：它标准化了什么、FDE 什么时候会去用它，以及它和你亲手搭的原生工具调用循环是什么关系。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。

> **这是一个概念 notebook，没有代码要跑。** 本仓库的主张是*原生 API，讲模式不讲封装*；MCP 是一个协议/封装层，所以我们只讲它是什么、什么时候值得用，而不从零写一个 MCP server。你已经在 `01`–`03` 里亲手搭过工具循环；这里就是那个循环与更大生态相遇的地方。

## MCP 解决的问题

在 `02-tool-design` 里，你把每个工具写成一个 Python 函数加一个 JSON schema，直接接进*你的* agent 循环。用来学习再合适不过，对一个应用来说也够用。但把它规模化，就会出现一个痛苦的 M×N 问题：

- 你有 **M 个 AI 应用**（一个聊天机器人、一个 IDE 助手、一个内部运维工具）。
- 你想让它们触达 **N 个系统**（GitHub、Postgres、Google Drive、你的工单系统）。
- 天真地做，就是 **M×N** 个定制集成：每个应用都用自己的方式重新实现每一个工具。

**MCP（Model Context Protocol）** 是一个开放标准，由 Anthropic 提出、如今已被广泛采用，它把 M×N 变成 M+N。每个系统把自己的能力*一次性*暴露为一个 **MCP server**；每个应用是一个 **MCP client**，可以和*任何* server 通信。官方的类比是：**MCP 之于 AI 应用就像 USB-C**，一个标准接口，取代每台设备一根定制线。

## 它如何映射到你已经搭好的循环

MCP 没有取代你学过的任何东西；它*把工具所需的同样三样东西的打包方式标准化了*。你亲手搭的一切，在协议里都有一个命名的位置：

| 你亲手搭的（nb 01–03） | MCP 里叫什么 | 它标准化了什么 |
|---|---|---|
| 一个工具 = 函数 + JSON schema | **Tool**（在 server 上） | 模型看到的工具定义 |
| 你的 `HANDLERS`/`IMPLS` 分发 | 由 **server** 运行它 | 执行发生的位置（现在是进程外） |
| 你塞进 prompt 的块/文档 | **Resource** | server 可以暴露的只读上下文 |
| 一个可复用的 prompt 模板 | **Prompt** | 具名、可分享的 prompt 脚手架 |
| 你调用 API 的 agent 循环 | **Client**（宿主应用内部） | 应用如何发现并调用 server |

你学到的机制正是 MCP 底下在跑的东西：模型仍然发出一个工具调用请求，某个东西执行它，结果返回。MCP 只是说「这是一种标准的方式来描述和传输它，于是一个写一次的工具就能在 Claude Desktop、VS Code、Cursor、ChatGPT 和你自己的应用里工作，而无需重写。

## FDE 什么时候会去用 MCP（以及什么时候不用）

**这些情况去用它：**
- 你要集成的工具**已经有 MCP server**（GitHub、Slack、Postgres、文件系统：一个庞大且在增长的目录）。免费集成胜过自己写。
- 你希望工具能在**多个宿主应用**里使用：写一次 server，插进 Claude、Cursor、你的应用。
- 你身处一个 **MCP 原生宿主**（Claude Desktop/Code、VS Code）里，想扩展它。

**这些情况跳过它（保留原生循环）：**
- 你在一个服务里只有**几个应用专属工具**。M=1 时 M×N 问题根本不存在，而一个直接的函数 + schema 比跑一个 server 更简单。
- 你需要对执行、延迟或精确的工具契约**有紧密控制**。间接层是有代价的。
- 你在**学习**，这也是为什么整个仓库都先教你手搭循环。在你了解 MCP 抽象的是什么之前，你没法评估它的抽象有没有帮助。

判断标准和本仓库里每一个关于框架的问题一样：**先把原生模式搞懂，当集成面（而不是 demo）足以证明它值得时，再采用这个标准。**

## 安全提示：这变成了更大的攻击面

一旦你的 agent 能触达任意的 MCP server（有些由第三方编写），工具本身就成了不可信输入。一个恶意或被攻陷的 server 可以返回精心构造的内容来劫持你的模型（间接 prompt 注入），或者诱导越权的动作（过度代理）。这些正是下一节（`07-security`）要讲的风险。MCP 让工具生态更大、更强，这使信任边界纪律变得更重要，而不是更不重要。

**生态故事里的下一站：** MCP 把你的 agent 连接到外部*系统*；配套的概念，[`05-skills-and-progressive-disclosure`](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/05-agents/05-skills-and-progressive-disclosure.zh.ipynb)，打包的是使用它们的*流程性知识*，按需加载，所以在被触发之前几乎不占上下文。工具负责*行动*，MCP 负责*连接*，Skills 负责*打包*。

**延伸阅读：** [MCP 文档](https://modelcontextprotocol.io/)（规范、server/client 指南、公开 server 目录）。

## 实践与反模式

| ✅ 该做 | ❌ 反模式 |
|---|---|
| 先学原生工具循环；当*集成面*足以证明它值当时再采用 MCP | 在你理解它所封装的循环之前就去用这个协议 |
| 用现成的 MCP server（GitHub、Slack、Postgres……）换取免费集成 | 为每个应用重新实现每个集成（M×N 陷阱） |
| 把第三方 MCP server 的输出当作不可信输入（见第 07 节） | 因为「它是个工具」就信任某个公开 server 返回的内容 |
| 严格限定每个 server 的权限；对重要动作加人工门禁 | 用宽泛权限把 agent 接到任意 server 上 |
| 少数几个应用专属工具就保留原生函数 + schema | 为一个服务里的一个工具搭一个 MCP server |

（贯穿整个技术栈的完整清单见 [docs/best-practices-and-anti-patterns.md](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/docs/best-practices-and-anti-patterns.md)。）

## 练习（不写代码，只做推理）

1. **用你自己的场景数一遍 M×N。** 说出你工作里的 3 个 AI 应用和 3 个后端系统。不用 MCP 有多少个定制集成？用了呢？让这个标准变得值得的交叉点在哪里？
2. **画出映射。** 从 `02-tool-design` 里挑一个工具，描述它如何作为一个 MCP *server* 暴露：Tool 是什么、Resource（如果有）是什么、谁是 client？
3. **做个判断。** 对你的结业项目（第 12 节）：你会用 MCP 还是一个原生工具循环？写出你会给面试官的两句话理由。推理比答案更重要。
4. **给一个公开 server 做威胁建模。** 你加了一个第三方的「web search」MCP server。列出它返回的内容攻击你的 agent 的两种方式，以及你如何分别缓解。（学完 `07-security` 后再回来看看。）